# **Taller 06: Reducción de Dimensiones**
### Inteligencia Artificial ELP 8012 - Universidad del Norte - Profesor: Eduardo Zurek, PH.D. 
### **Grupo:** Natalia Carpintero, Paula Núñez e Isabella Arrieta.

Aplicar y comparar distintas técnicas de reducción de dimensiones (**PCA, MDS Clásico, MDS Métrico, MDS No-métrico y UMAP**) sobre datos de acelerómetro y giroscopio de teléfono y smartwatch, provenientes del **WISDM Smartphone and Smartwatch Activity and Biometrics Dataset**.

Para cada uno de los 4 casos (`phone-accel`, `phone-gyro`, `watch-accel`, `watch-gyro`):

1. Se seleccionan aleatoriamente **3 voluntarios** (de un total de 50, codificados de 1600 a 1650, sin el 1614).
2. Se construye, para cada voluntario, una matriz con las columnas `xyzbins` (`X0...X9`, `Y0...Y9`, `Z0...Z9`) y se define el *ground truth* con la columna `ACTIVITY`.
3. Se proyectan los datos a 2 dimensiones con cada técnica, coloreando por `ACTIVITY`.
4. Se repite el proceso uniendo las 3 muestras en una sola matriz por caso.


## 1. Descarga y ubicación del dataset

El dataset se descarga desde el repositorio UCI:

> https://archive.ics.uci.edu/dataset/507/wisdm+smartphone+and+smartwatch+activity+and+biometrics+dataset

Descomprima el archivo `wisdm-dataset.zip` en la misma carpeta de este notebook. La estructura esperada es:

```
wisdm-dataset/
└── arff_files/
    ├── phone/
    │   ├── accel/
    │   │   ├── data_1600_accel_phone.arff
    │   │   ├── data_1601_accel_phone.arff
    │   │   └── ...
    │   └── gyro/
    │       ├── data_1600_gyro_phone.arff
    │       └── ...
    └── watch/
        ├── accel/
        │   ├── data_1600_accel_watch.arff
        │   └── ...
        └── gyro/
            ├── data_1600_gyro_watch.arff
            └── ...
```

Cada archivo `.arff` contiene, entre otras columnas, las 30 columnas `xyzbins` (`X0..X9`, `Y0..Y9`, `Z0..Z9`) —histogramas de las lecturas del sensor— y la columna de clase con la actividad realizada (`ACTIVITY` o `class`, según la versión del dataset).

Ajuste la variable `DATA_DIR` en la siguiente celda según donde haya descomprimido el dataset.


### 📊 **Optimizaciones de velocidad**

El notebook ha sido optimizado para reducir el tiempo de ejecución sin eliminar muestras. Los cambios incluyen:

- ✅ **MDS**: Una inicialización y máximo 300 iteraciones para mantener el análisis completo viable
- ✅ **MDS paralelo**: Uso de todos los núcleos disponibles mediante `n_jobs=-1`
- ✅ **UMAP**: Menos épocas (200 en lugar de 500) y optimización de vecinos
- ✅ **Indicadores de progreso**: Seguimiento visual del avance

**Tiempo estimado:** variable según el equipo; MDS puede tardar varios minutos por caso.



In [2]:
# Si hace falta, instale las dependencias (descomente en su entorno local)
%pip install -q pandas numpy scipy scikit-learn matplotlib seaborn umap-learn


Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.io import arff as scipy_arff
from scipy.spatial.distance import pdist, squareform

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import MDS

import umap

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42  # semilla para reproducibilidad de los algoritmos (no de la selección de voluntarios)
np.random.seed(RANDOM_STATE)


In [12]:
# Ruta base donde se descomprimió el dataset.
# Se prueban las dos ubicaciones habituales según desde dónde se abrió el notebook.
DATA_DIR_CANDIDATES = [
    Path("./wisdm-dataset/arff_files"),
    Path("./Taller6-Reduccion-de-dimensiones/wisdm-dataset/arff_files"),
]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "No se encontró 'wisdm-dataset/arff_files'. "
        "Ajuste DATA_DIR_CANDIDATES a la ubicación del dataset."
    )
print(f"Dataset encontrado en: {DATA_DIR.resolve()}")

DEVICES = ["phone", "watch"]
SENSORS = ["accel", "gyro"]
CASES = [(d, s) for d in DEVICES for s in SENSORS]  # phone-accel, phone-gyro, watch-accel, watch-gyro

def arff_path(subject_id, device, sensor):
    """Construye la ruta al archivo .arff de un voluntario/dispositivo/sensor."""
    fname = f"data_{subject_id}_{sensor}_{device}.arff"
    return DATA_DIR / device / sensor / fname

Dataset encontrado en: C:\Users\USUARIO\Downloads\Taller6-Reduccion-de-dimensiones\wisdm-dataset\arff_files


## 2. Selección aleatoria de 3 voluntarios

Los voluntarios están codificados de **1600 a 1650**, sin el **1614** (50 voluntarios en total). Seleccionamos 3 de ellos aleatoriamente.


In [13]:
ALL_SUBJECTS = [s for s in range(1600, 1651) if s != 1614]
assert len(ALL_SUBJECTS) == 50, "Se esperaban 50 voluntarios"

SELECTED_SUBJECTS = random.sample(ALL_SUBJECTS, 3)
print("Voluntarios seleccionados:", SELECTED_SUBJECTS)


Voluntarios seleccionados: [1600, 1627, 1647]


In [14]:
# ==================== DEFINICIÓN DE load_case ====================

BIN_COL_PATTERN = re.compile(r"^[XYZ][0-9]$")

def load_case(subject_id, device, sensor):
    """Carga un archivo .arff y devuelve (X, y):
        X: DataFrame con las columnas xyzbins (X0..X9, Y0..Y9, Z0..Z9)
        y: Serie con la actividad (ground truth)"""
    path = arff_path(subject_id, device, sensor)
    data, meta = scipy_arff.loadarff(path)
    df = pd.DataFrame(data)
    # scipy.io.arff conserva las comillas del encabezado en los nombres.
    df.columns = [str(col).strip('"') for col in df.columns]

    # Decodificar columnas de tipo bytes (típico en ARFF con atributos nominales)
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(lambda v: v.decode("utf-8") if isinstance(v, bytes) else v)

    # Identificar la columna de la clase (ACTIVITY / class)
    label_col = None
    for candidate in ["ACTIVITY", "class", "Class", "activity"]:
        if candidate in df.columns:
            label_col = candidate
            break
    if label_col is None:
        label_col = df.columns[-1]  # último recurso: última columna

    bin_cols = [c for c in df.columns if BIN_COL_PATTERN.match(c)]
    assert len(bin_cols) == 30, f"Se esperaban 30 columnas xyzbins, se encontraron {len(bin_cols)}: {bin_cols}"

    X = df[bin_cols].astype(float)
    y = df[label_col].astype(str)
    y.name = "ACTIVITY"
    
    return X, y


## 3. Carga de datos

Definimos una función que, dado un voluntario, dispositivo y sensor, carga el archivo `.arff` correspondiente y devuelve una matriz con las **30 columnas xyzbins** (`X0..X9`, `Y0..Y9`, `Z0..Z9`) junto con la columna `ACTIVITY` (ground truth).


In [15]:
def load_case_safe(subject_id, device, sensor):
    """Igual que load_case, pero captura errores de archivo faltante/corrupto."""
    try:
        return load_case(subject_id, device, sensor)
    except FileNotFoundError:
        print(f"[Aviso] No se encontró el archivo para sujeto {subject_id}, {device}-{sensor}.")
        return None, None


## 4. Funciones de reducción de dimensiones

Implementamos las 5 técnicas solicitadas:

- **PCA**: `sklearn.decomposition.PCA`
- **MDS Clásico** (Torgerson, vía eigendescomposición de la matriz de distancias doblemente centrada) — implementado manualmente porque `scikit-learn` sólo ofrece la variante iterativa (SMACOF).
- **MDS Métrico** (SMACOF, `metric=True`)
- **MDS No-métrico** (SMACOF, `metric=False`)
- **UMAP**: `umap.UMAP`

Antes de reducir, estandarizamos las variables (media 0, varianza 1) para que ningún bin domine por escala.


In [16]:
def classical_mds(X, n_components=2):
    """MDS clásico (Torgerson) vía eigendescomposición de la matriz de
    distancias euclidianas doblemente centrada (equivalente a PCoA)."""
    D = squareform(pdist(X, metric="euclidean"))
    n = D.shape[0]
    D2 = D ** 2
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D2 @ J

    eigvals, eigvecs = np.linalg.eigh(B)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    pos_idx = eigvals > 0
    L = np.diag(np.sqrt(eigvals[pos_idx][:n_components]))
    V = eigvecs[:, pos_idx][:, :n_components]
    return V @ L


def reduce_dimensions(X, random_state=RANDOM_STATE, verbose=True):
    """Aplica las 5 técnicas de reducción de dimensiones y devuelve un
    diccionario {nombre_tecnica: embedding_2D}."""
    Xs = StandardScaler().fit_transform(X)
    embeddings = {}

    if verbose:
        print(f"Reduciendo {len(X)} muestras de {X.shape[1]} dimensiones...")

    # PCA (muy rápido)
    if verbose: print("  • PCA...", end=" ", flush=True)
    embeddings["PCA"] = PCA(n_components=2, random_state=random_state).fit_transform(Xs)
    if verbose: print("✓")

    # MDS Clásico (muy rápido)
    if verbose: print("  • MDS Clásico...", end=" ", flush=True)
    embeddings["MDS Clásico"] = classical_mds(Xs, n_components=2)
    if verbose: print("✓")

    # MDS Métrico: usa todos los datos con una configuración más rápida.
    if verbose: print("  • MDS Métrico...", end=" ", flush=True)
    embeddings["MDS Métrico"] = MDS(
        n_components=2, metric=True, dissimilarity="euclidean",
        random_state=random_state, normalized_stress="auto", n_init=1, max_iter=300, n_jobs=-1
    ).fit_transform(Xs)
    if verbose: print("✓")

    # MDS No-métrico: usa todos los datos con una configuración más rápida.
    if verbose: print("  • MDS No-métrico...", end=" ", flush=True)
    embeddings["MDS No-métrico"] = MDS(
        n_components=2, metric=False, dissimilarity="euclidean",
        random_state=random_state, normalized_stress="auto", n_init=1, max_iter=300, n_jobs=-1
    ).fit_transform(Xs)
    if verbose: print("✓")

    # UMAP (OPTIMIZADO: menos vecinos y épocas)
    if verbose: print("  • UMAP...", end=" ", flush=True)
    embeddings["UMAP"] = umap.UMAP(
        n_components=2, random_state=random_state,
        n_neighbors=min(15, len(X)-1),  # menos vecinos
        n_epochs=200,  # menos épocas (por defecto es 500)
        verbose=False
    ).fit_transform(Xs)
    if verbose: print("✓")

    if verbose: print("Listo!\n")
    return embeddings


In [17]:
def plot_embeddings(embeddings, y, title, figsize=(24, 4.5)):
    """Grafica en una fila los 5 embeddings 2D, coloreados por ACTIVITY."""
    techniques = list(embeddings.keys())
    fig, axes = plt.subplots(1, len(techniques), figsize=figsize)
    fig.suptitle(title, fontsize=14, y=1.05)

    activities = sorted(y.unique())
    palette = sns.color_palette("tab20", n_colors=len(activities))
    color_map = dict(zip(activities, palette))

    for ax, tech in zip(axes, techniques):
        emb = embeddings[tech]
        for act in activities:
            mask = (y == act).values
            ax.scatter(emb[mask, 0], emb[mask, 1], s=12, color=color_map[act], label=act, alpha=0.8)
        ax.set_title(tech, fontsize=11)
        ax.set_xlabel("Dim 1")
        ax.set_ylabel("Dim 2")

    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5),
               fontsize=8, title="ACTIVITY", ncol=1)
    plt.tight_layout()
    plt.show()


## 5. Análisis individual por muestra

Para cada uno de los 3 voluntarios seleccionados y cada uno de los 4 casos (`phone-accel`, `phone-gyro`, `watch-accel`, `watch-gyro`), cargamos la matriz `xyzbins`.

Los datos crudos (`X`, `y`) de cada muestra se guardan para poder combinarlos en la siguiente fase.

**Ejecución:** Ejecuta esta celda para cargar todos los datos.



In [18]:
# FASE 1: Cargar datos individuales por voluntario (SIN GRÁFICAS)
print("=" * 70)
print("FASE 1: Cargando datos individuales por voluntario")
print("=" * 70)

raw_data = {}  # Diccionario para almacenar datos crudos

for i, subject_id in enumerate(SELECTED_SUBJECTS):
    print(f"\nVoluntario {i+1}/3: {subject_id}")
    for device, sensor in CASES:
        case_name = f"{device}-{sensor}"
        print(f"  • Cargando {case_name}...", end=" ", flush=True)
        X, y = load_case_safe(subject_id, device, sensor)
        if X is None:
            print("❌")
            continue
        raw_data[(subject_id, case_name)] = (X, y)
        print(f"✓ ({len(y)} muestras)")

print("\n" + "=" * 70)
print("✓ Fase 1 completada")
print("=" * 70)


FASE 1: Cargando datos individuales por voluntario

Voluntario 1/3: 1600
  • Cargando phone-accel... ✓ (321 muestras)
  • Cargando phone-gyro... ✓ (321 muestras)
  • Cargando watch-accel... ✓ (327 muestras)
  • Cargando watch-gyro... ✓ (327 muestras)

Voluntario 2/3: 1627
  • Cargando phone-accel... ✓ (775 muestras)
  • Cargando phone-gyro... ✓ (321 muestras)
  • Cargando watch-accel... ✓ (324 muestras)
  • Cargando watch-gyro... ✓ (324 muestras)

Voluntario 3/3: 1647
  • Cargando phone-accel... ✓ (409 muestras)
  • Cargando phone-gyro... ✓ (405 muestras)
  • Cargando watch-accel... ✓ (327 muestras)
  • Cargando watch-gyro... ✓ (327 muestras)

✓ Fase 1 completada


## 6. Reducción de dimensiones individual

Aplicamos PCA, MDS Clásico, MDS Métrico, MDS No-métrico y UMAP a cada matriz individual guardada en la fase anterior. Las gráficas se mostrarán después de terminar también el procesamiento combinado.



In [19]:
print("\n" + "=" * 70)
print("FASE 2: Aplicando técnicas a cada muestra individual")
print("=" * 70 + "\n")

individual_results = {}

for (subject_id, case_name), (X, y) in raw_data.items():
    print(f"Procesando voluntario {subject_id}, {case_name}...", flush=True)
    individual_results[(subject_id, case_name)] = (
        reduce_dimensions(X, verbose=True),
        y
    )

print("=" * 70)
print("✓ Fase 2 completada")
print("=" * 70)



FASE 2: Aplicando técnicas a cada muestra individual

Procesando voluntario 1600, phone-accel...
Reduciendo 321 muestras de 30 dimensiones...
  • PCA... ✓
  • MDS Clásico... ✓
  • MDS Métrico... ✓
  • MDS No-métrico... ✓
  • UMAP... ✓
Listo!

Procesando voluntario 1600, phone-gyro...
Reduciendo 321 muestras de 30 dimensiones...
  • PCA... ✓
  • MDS Clásico... ✓
  • MDS Métrico... ✓
  • MDS No-métrico... ✓
  • UMAP... ✓
Listo!

Procesando voluntario 1600, watch-accel...
Reduciendo 327 muestras de 30 dimensiones...
  • PCA... ✓
  • MDS Clásico... ✓
  • MDS Métrico... ✓
  • MDS No-métrico... ✓
  • UMAP... ✓
Listo!

Procesando voluntario 1600, watch-gyro...
Reduciendo 327 muestras de 30 dimensiones...
  • PCA... ✓
  • MDS Clásico... ✓
  • MDS Métrico... ✓
  • MDS No-métrico... ✓
  • UMAP... ✓
Listo!

Procesando voluntario 1627, phone-accel...
Reduciendo 775 muestras de 30 dimensiones...
  • PCA... ✓
  • MDS Clásico... ✓
  • MDS Métrico... 

KeyboardInterrupt: 

## 7. Análisis combinado por caso

Para cada caso (`phone-accel`, `phone-gyro`, `watch-accel`, `watch-gyro`), unimos la información de los 3 voluntarios en una sola matriz y aplicamos las 5 técnicas de reducción de dimensiones.

**Nota:** Esta fase es la más costosa computacionalmente (20-30 minutos). Los indicadores de progreso mostrarán el avance de cada técnica.

**Ejecución:** Ejecuta esta celda después de la reducción individual.



In [ ]:
print("\n" + "=" * 70)
print("FASE 3: Procesando análisis combinado por caso (SIN GRÁFICAS)")
print("=" * 70 + "\n")

combined_results = {}

for device, sensor in CASES:
    case_name = f"{device}-{sensor}"
    print(f"Procesando {case_name}...", flush=True)

    Xs_list, ys_list = [], []
    for subject_id in SELECTED_SUBJECTS:
        if (subject_id, case_name) in raw_data:
            X_s, y_s = raw_data[(subject_id, case_name)]
            Xs_list.append(X_s)
            ys_list.append(y_s)

    if not Xs_list:
        print(f"  [Aviso] No hay datos disponibles\n")
        continue

    X_combined = pd.concat(Xs_list, axis=0, ignore_index=True)
    y_combined = pd.concat(ys_list, axis=0, ignore_index=True)

    embeddings = reduce_dimensions(X_combined, verbose=True)
    combined_results[case_name] = (embeddings, y_combined)

print("=" * 70)
print("✓ Fase 3 completada")
print("=" * 70)



FASE 2: Procesando análisis combinado por caso (SIN GRÁFICAS)

Procesando phone-accel...
Reduciendo 1754 muestras de 30 dimensiones...
  • PCA... ✓
  • MDS Clásico... ✓
  • MDS Métrico... ✓
  • MDS No-métrico... 

KeyboardInterrupt: 

## 8. Visualización de resultados individuales

Esta celda muestra las 12 figuras individuales: una por cada voluntario y caso. Cada figura contiene las cinco técnicas de reducción, coloreadas por `ACTIVITY`.



In [ ]:
print("=" * 70)
print("FASE 4: Visualizando resultados individuales")
print("=" * 70 + "\n")

for (subject_id, case_name), (embeddings, y) in individual_results.items():
    plot_embeddings(
        embeddings,
        y,
        title=f"Voluntario {subject_id}: {case_name} (n={len(y)})"
    )

print("\n" + "=" * 70)
print("✓ Visualización individual completada")
print("=" * 70)


## 10. Discusión

Después de ejecutar las dos celdas de visualización, responda las siguientes preguntas observando sus propias figuras.

### ¿Qué actividades se diferencian mejor en dos dimensiones?

En general, en este tipo de datos de acelerómetro/giroscopio suelen separarse mejor las actividades con **patrones de movimiento periódicos y de alta amplitud**, como `WALKING`, `JOGGING` o `STAIRS`, ya que generan histogramas de bins distintos a los de actividades estáticas.

### ¿Qué actividades se superponen en dos dimensiones?

Suelen superponerse actividades **estáticas o con movimientos finos y similares entre sí**, como `SITTING`, `STANDING`, `TYPING` o `WRITING`, dado que producen señales de baja amplitud y variabilidad parecida entre los ejes X, Y, Z.

### Comparación entre técnicas

- **PCA** captura varianza global de forma lineal y rápida, pero puede no separar bien clases con relaciones no lineales.
- **MDS Clásico** utiliza las distancias euclidianas y suele producir resultados similares a PCA.
- **MDS Métrico** y **No-métrico** intentan preservar las distancias, o su orden, entre puntos, pero son más costosos computacionalmente.
- **UMAP** captura estructura local no lineal y puede producir agrupamientos más compactos.

Las conclusiones definitivas deben basarse en las gráficas obtenidas con los tres voluntarios seleccionados.



## 9. Visualización de resultados combinados

Esta celda muestra las 4 figuras combinadas, una por cada caso. Cada figura contiene PCA, MDS Clásico, MDS Métrico, MDS No-métrico y UMAP, coloreados por `ACTIVITY`.



In [ ]:
print("=" * 70)
print("FASE 5: Visualizando resultados combinados")
print("=" * 70 + "\n")

for case_name, (embeddings, y_combined) in combined_results.items():
    plot_embeddings(
        embeddings,
        y_combined,
        title=f"Combinado: 3 voluntarios - {case_name} (n={len(y_combined)})"
    )

print("\n" + "=" * 70)
print("✓ Visualización combinada completada")
print("=" * 70)
